# Belief Elicitation Framework Demo

This notebook demonstrates how to elicit beliefs from LLMs during economic games using proper scoring rules (PSR), with **pre-decision** belief elicitation in forked conversation branches.

## Key Concepts

1. **Pre-decision elicitation**: Beliefs are elicited *before* the LLM chooses its action, enabling causal identification of the belief→action relationship.

2. **Forked branches**: Belief questions are asked in a deep-copied conversation branch that is discarded after extraction — the main game conversation is never contaminated.

3. **Dual mode**: Supports both *incentivized* (PSR framing) and *direct ask* (no framing) elicitation.

## Games (all have no dominant strategy — beliefs matter)

| Game | Strategic Uncertainty |
|------|---------------------|
| Prisoner's Dilemma | Beliefs about opponent's Push/Pull |
| Stag Hunt | Beliefs about partner's Stag/Hare |
| p-Beauty Contest | Higher-order beliefs about others' choices |
| Ultimatum (Proposer) | Beliefs about responder acceptance |
| First-Price Auction | Beliefs about opponent's bid |

## Setup

In [1]:
import os
import sys
import json
import openai
from datetime import datetime

# Add project root to path (adjust if needed)
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('')), ''))

from belief_elicitation import (
    # Runner creation
    create_belief_aware_runner,

    # Game runners
    run_pd_with_beliefs,
    run_stag_hunt_with_beliefs,
    run_beauty_contest_with_beliefs,
    run_first_price_auction_with_beliefs,
    run_ultimatum_proposer_with_beliefs,

    # Config functions
    get_prisoners_dilemma_config,
    get_stag_hunt_config,
    get_beauty_contest_config,
    get_first_price_auction_config,
    get_ultimatum_proposer_config,
    list_available_games,

    # Opponent strategies
    OpponentStrategy,
    OpponentStrategyType,

    # Analysis
    extract_beliefs_summary,
    extract_choices_from_records,
)

In [2]:
BASE_URL = "http://35.220.164.252:3888/v1/"
API_KEY = "sk-c8j8scGHgCgeEkDvKetBLee00aDrhOAGuJ4WRctUDvmNXjPp"
MODEL = 'gpt-4.1-mini'

client = openai.OpenAI(api_key=API_KEY, base_url=BASE_URL)

runner, get_response, update_messages = create_belief_aware_runner(
    client=client,
    model=MODEL,
    system_message="You are a helpful assistant.",
    time_limit=60
)

print("Available games:", list_available_games())

Available games: ['prisoners_dilemma', 'stag_hunt', 'ultimatum_proposer', 'beauty_contest', 'first_price_auction']


## Example 1: Prisoner's Dilemma

Multi-round PD with configurable opponent strategy. Beliefs about P(Push) vs P(Pull) are elicited **pre-decision** each round.

In [3]:
# Run PD: 3 instances, 3 rounds, opponent randomly cooperates 30% of the time
records_pd = run_pd_with_beliefs(
    runner=runner,
    get_response_func=get_response,
    n_instances=3,
    n_rounds=3,
    opponent_strategy=OpponentStrategy(
        strategy_type=OpponentStrategyType.RANDOM,
        p_cooperate=0.3,  # opponent Pushes 30% of the time
        cooperate_label="Push",
        defect_label="Pull"
    ),
    print_except=True
)

  0%|          | 0/3 [00:00<?, ?it/s]

 33%|███▎      | 1/3 [00:25<00:51, 25.57s/it]

 67%|██████▋   | 2/3 [00:45<00:22, 22.34s/it]

100%|██████████| 3/3 [01:16<00:00, 26.07s/it]

100%|██████████| 3/3 [01:16<00:00, 25.39s/it]

In [4]:
# Analyze PD results: beliefs vs actions per instance
for i in range(len(records_pd['choices'])):
    choices = records_pd['choices'][i]
    opp = records_pd['opponent_actions'][i]
    beliefs = records_pd['beliefs'][i]

    print(f"--- Instance {i+1} ---")
    for r, (c, b) in enumerate(zip(choices, beliefs), 1):
        belief_data = b.get('beliefs', {})
        # Get P(Push) for this round
        key = f'opponent_action_r{r}'
        p_push = belief_data.get(key, {}).get('Push', '?')
        print(f"  Round {r}: Belief P(Push)={p_push:.2f}" if isinstance(p_push, float) else f"  Round {r}: Belief={belief_data}")
        print(f"           Action={c}, Opponent={opp[r-1] if r-1 < len(opp) else '?'}")
    print()

--- Instance 1 ---
  Round 1: Belief P(Push)=0.40
           Action=Push, Opponent=Pull
  Round 2: Belief P(Push)=0.30
           Action=Pull, Opponent=Push
  Round 3: Belief P(Push)=0.50
           Action=Pull, Opponent=Pull

--- Instance 2 ---
  Round 1: Belief P(Push)=0.50
           Action=Push, Opponent=Push
  Round 2: Belief P(Push)=0.70
           Action=Push, Opponent=Pull
  Round 3: Belief P(Push)=0.30
           Action=Pull, Opponent=Pull

--- Instance 3 ---
  Round 1: Belief P(Push)=0.40
           Action=Pull, Opponent=Push
  Round 2: Belief P(Push)=0.40
           Action=Pull, Opponent=Push
  Round 3: Belief P(Push)=0.35
           Action=Pull, Opponent=Push



In [5]:
# Summary statistics across instances
belief_summary = extract_beliefs_summary(records_pd)

print("Belief Summary Statistics (PD):")
for q_id, stats in sorted(belief_summary.items()):
    if isinstance(stats, dict) and 'mean' in stats:
        print(f"  {q_id}: mean={stats['mean']:.3f}, std={stats['std']:.3f}, n={stats['n']}")
    else:
        for outcome, s in stats.items():
            print(f"  {q_id}/{outcome}: mean={s['mean']:.3f}, std={s['std']:.3f}")

Belief Summary Statistics (PD):
  opponent_action_r1/Push: mean=0.433, std=0.047
  opponent_action_r1/Pull: mean=0.567, std=0.047
  opponent_action_r2/Push: mean=0.467, std=0.170
  opponent_action_r2/Pull: mean=0.533, std=0.170
  opponent_action_r3/Push: mean=0.383, std=0.085
  opponent_action_r3/Pull: mean=0.617, std=0.085


## Example 2: Stag Hunt

Coordination game with two pure NE: (Stag,Stag) is payoff-dominant, (Hare,Hare) is risk-dominant. The LLM's belief about the partner's choice determines which equilibrium it selects.

In [6]:
# Stag Hunt with Tit-for-Tat opponent
records_sh = run_stag_hunt_with_beliefs(
    runner=runner,
    get_response_func=get_response,
    n_instances=3,
    n_rounds=3,
    opponent_strategy=OpponentStrategy(
        strategy_type=OpponentStrategyType.TIT_FOR_TAT,
        cooperate_label="Stag",
        defect_label="Hare"
    ),
    print_except=True
)

for i in range(len(records_sh['choices'])):
    choices = records_sh['choices'][i]
    opp = records_sh['opponent_actions'][i]
    beliefs = records_sh['beliefs'][i]
    print(f"--- Instance {i+1} ---")
    for r, (c, b) in enumerate(zip(choices, beliefs), 1):
        bd = b.get('beliefs', {})
        key = f'partner_action_r{r}'
        p_stag = bd.get(key, {}).get('Stag', '?')
        p_str = f"{p_stag:.2f}" if isinstance(p_stag, float) else str(p_stag)
        print(f"  Round {r}: P(Stag)={p_str}, Chose={c}, Opp={opp[r-1] if r-1 < len(opp) else '?'}")
    print()

  0%|          | 0/3 [00:00<?, ?it/s]

 33%|███▎      | 1/3 [00:29<00:59, 29.96s/it]

 67%|██████▋   | 2/3 [01:02<00:31, 31.43s/it]

100%|██████████| 3/3 [01:27<00:00, 28.63s/it]

100%|██████████| 3/3 [01:27<00:00, 29.24s/it]

--- Instance 1 ---
  Round 1: P(Stag)=0.70, Chose=Stag, Opp=Stag
  Round 2: P(Stag)=0.80, Chose=Stag, Opp=Stag
  Round 3: P(Stag)=0.80, Chose=Stag, Opp=Stag

--- Instance 2 ---
  Round 1: P(Stag)=0.65, Chose=Stag, Opp=Stag
  Round 2: P(Stag)=0.80, Chose=Stag, Opp=Stag
  Round 3: P(Stag)=0.90, Chose=Stag, Opp=Stag

--- Instance 3 ---
  Round 1: P(Stag)=0.40, Chose=Stag, Opp=Stag
  Round 2: P(Stag)=0.80, Chose=Stag, Opp=Stag
  Round 3: P(Stag)=0.90, Chose=Stag, Opp=Stag



## Example 3: p-Beauty Contest

Tests higher-order beliefs. Each player picks a number in [0,100]; the winner is closest to p × average. Nash equilibrium is 0, but real play depends on depth of strategic reasoning.

In [7]:
# Beauty Contest: 3 players, p=2/3, 3 rounds with feedback
records_bc = run_beauty_contest_with_beliefs(
    runner=runner,
    get_response_func=get_response,
    n_instances=2,
    n_rounds=3,
    p=2/3,
    n_players=3,
    print_except=True
)

for i in range(len(records_bc['choices'])):
    choices = records_bc['choices'][i]
    results = records_bc['round_results'][i]
    beliefs = records_bc['beliefs'][i]
    print(f"--- Instance {i+1} ---")
    for r, (c, res, b) in enumerate(zip(choices, results, beliefs), 1):
        bd = b.get('beliefs', {})
        key = f'expected_others_avg_r{r}'
        e_avg = bd.get(key, '?')
        e_str = f"{e_avg:.1f}" if isinstance(e_avg, float) else str(e_avg)
        print(f"  Round {r}: E[others' avg]={e_str}, Chose={c:.1f}, "
              f"Actual avg={res['average']:.1f}, Target={res['target']:.1f}")
    print()

  0%|          | 0/2 [00:00<?, ?it/s]

 50%|█████     | 1/2 [00:27<00:27, 27.24s/it]

100%|██████████| 2/2 [00:47<00:00, 23.11s/it]

100%|██████████| 2/2 [00:47<00:00, 23.73s/it]

--- Instance 1 ---
  Round 1: E[others' avg]=50.0, Chose=44.0, Actual avg=31.7, Target=21.1
  Round 2: E[others' avg]=30.0, Chose=22.0, Actual avg=37.8, Target=25.2
  Round 3: E[others' avg]=34.0, Chose=25.0, Actual avg=25.6, Target=17.1

--- Instance 2 ---
  Round 1: E[others' avg]=50.0, Chose=44.0, Actual avg=43.6, Target=29.1
  Round 2: E[others' avg]=43.0, Chose=30.0, Actual avg=19.9, Target=13.3
  Round 3: E[others' avg]=15.0, Chose=14.0, Actual avg=36.4, Target=24.2



## Example 4: First-Price Sealed-Bid Auction

Bayesian game: LLM gets a private valuation v ~ Uniform[0,100] and submits a bid. The BNE strategy is bid = v/2. We elicit beliefs about the opponent's bid.

In [8]:
# First-Price Auction: opponent uses BNE strategy (bid = v/2)
records_auc = run_first_price_auction_with_beliefs(
    runner=runner,
    get_response_func=get_response,
    n_instances=5,
    print_except=True
)

print(f"{'Valuation':>10} {'Bid':>6} {'BNE(v/2)':>8} {'E[Opp Bid]':>10} {'Won':>5} {'Profit':>8}")
print("-" * 55)
for i in range(len(records_auc['bids'])):
    o = records_auc['outcomes'][i]
    beliefs = records_auc['beliefs'][i]
    e_opp = beliefs[0]['beliefs'].get('expected_opponent_bid', '?') if beliefs else '?'
    e_str = f"{e_opp:.1f}" if isinstance(e_opp, float) else str(e_opp)
    v = o['my_valuation']
    print(f"  ${v:6.1f}  ${o['my_bid']:5.1f}  ${v/2:7.1f}  ${e_str:>9}  {str(o['won']):>5}  ${o['profit']:7.1f}")

  0%|          | 0/5 [00:00<?, ?it/s]

 20%|██        | 1/5 [00:07<00:29,  7.49s/it]

 40%|████      | 2/5 [00:22<00:35, 11.72s/it]

 60%|██████    | 3/5 [00:32<00:22, 11.18s/it]

Ambiguous answer: Given that this is a first-price sealed-bid auction with valuations uniformly distributed from $0 to $100, a common strategy is to bid below your own valuation to maximize expected utility while remai
Cannot extract bid:
Given that this is a first-price sealed-bid auction with valuations uniformly distributed from $0 to $100, a common strategy is to bid below your own valuation to maximize expected utility while remai


 80%|████████  | 4/5 [00:49<00:13, 13.32s/it]

100%|██████████| 5/5 [00:55<00:00, 10.60s/it]

100%|██████████| 5/5 [00:55<00:00, 11.02s/it]

 Valuation    Bid BNE(v/2) E[Opp Bid]   Won   Profit
-------------------------------------------------------
  $   3.9  $  2.0  $    2.0  $     25.0  False  $    0.0
  $   4.5  $  3.0  $    2.3  $     25.0  False  $    0.0
  $  42.5  $ 21.0  $   21.2  $     25.0   True  $   21.5
  $  17.6  $  9.0  $    8.8  $     25.0  False  $    0.0
  $   1.3  $  1.0  $    0.6  $     25.0  False  $    0.0


## Example 5: Ultimatum Game (Proposer)

Proposer decides how to split $100. Beliefs about P(Accept | offer=$x) are elicited at multiple offer levels before the decision.

In [9]:
records_ult = run_ultimatum_proposer_with_beliefs(
    runner=runner,
    get_response_func=get_response,
    n_instances=3,
    print_except=True
)

offers = extract_choices_from_records(records_ult)
print(f"Offers made: {['$' + str(int(x)) for x in offers]}\n")

belief_summary = extract_beliefs_summary(records_ult)
print("Beliefs about acceptance probability:")
for q_id, stats in sorted(belief_summary.items()):
    if isinstance(stats, dict) and 'mean' in stats:
        print(f"  {q_id}: {stats['mean']*100:.1f}% (std={stats['std']*100:.1f}%, n={stats['n']})")

  0%|          | 0/3 [00:00<?, ?it/s]

 33%|███▎      | 1/3 [00:10<00:21, 10.94s/it]

 67%|██████▋   | 2/3 [00:21<00:10, 10.95s/it]

100%|██████████| 3/3 [00:31<00:00, 10.40s/it]

100%|██████████| 3/3 [00:31<00:00, 10.55s/it]

Offers made: ['$30', '$30', '$30']

Beliefs about acceptance probability:
  accept_prob_10: 60.0% (std=0.0%, n=3)
  accept_prob_20: 85.0% (std=0.0%, n=3)
  accept_prob_30: 95.0% (std=0.0%, n=3)
  accept_prob_40: 98.3% (std=0.5%, n=3)
  accept_prob_50: 99.3% (std=0.4%, n=3)


## Incentivized vs Direct Ask Comparison

Run the same game with PSR framing (incentivized) and without (direct ask) to see if incentive framing changes belief reports.

In [10]:
import random
random.seed(42)

# Same opponent for fair comparison
opp = OpponentStrategy(
    strategy_type=OpponentStrategyType.RANDOM,
    p_cooperate=0.5,
    cooperate_label="Push",
    defect_label="Pull"
)

# Incentivized (PSR framing)
random.seed(42)
rec_inc = run_pd_with_beliefs(
    runner=runner, get_response_func=get_response,
    n_instances=2, n_rounds=2,
    opponent_strategy=opp,
    belief_config=get_prisoners_dilemma_config(n_rounds=2, incentivized=True),
    print_except=True
)

# Direct ask (no framing)
random.seed(42)
rec_dir = run_pd_with_beliefs(
    runner=runner, get_response_func=get_response,
    n_instances=2, n_rounds=2,
    opponent_strategy=opp,
    belief_config=get_prisoners_dilemma_config(n_rounds=2, incentivized=False),
    print_except=True
)

print("Incentivized beliefs:")
for b in rec_inc['beliefs'][0]:
    print(f"  {b['point_id']}: {b['beliefs']}")

print("\nDirect-ask beliefs:")
for b in rec_dir['beliefs'][0]:
    print(f"  {b['point_id']}: {b['beliefs']}")

  0%|          | 0/2 [00:00<?, ?it/s]

 50%|█████     | 1/2 [00:20<00:20, 20.55s/it]

100%|██████████| 2/2 [00:37<00:00, 18.57s/it]

100%|██████████| 2/2 [00:37<00:00, 18.87s/it]

  0%|          | 0/2 [00:00<?, ?it/s]

 50%|█████     | 1/2 [00:16<00:16, 16.49s/it]

100%|██████████| 2/2 [00:32<00:00, 16.06s/it]

100%|██████████| 2/2 [00:32<00:00, 16.13s/it]

Incentivized beliefs:
  round_1_beliefs: {'opponent_action_r1': {'Push': 0.5, 'Pull': 0.5}}
  round_2_beliefs: {'opponent_action_r2': {'Push': 0.3, 'Pull': 0.7}}

Direct-ask beliefs:
  round_1_beliefs: {'opponent_action_r1': {'Push': 0.5, 'Pull': 0.5}}
  round_2_beliefs: {'opponent_action_r2': {'Push': 0.3, 'Pull': 0.7}}


## Pre-Decision Branching Mechanism

The critical design: beliefs are elicited in a **forked branch before the LLM acts**. The main conversation never sees the belief questions.

```
Main Conversation:
[System] → [Game Prompt] ──────────────────────→ [LLM Action] → [Feedback] → ...
                          │                       ↑
                          │ (deep copy, pre-action)│ (branch discarded)
                          ↓                       │
Belief Branch:      [Belief Q1] → [Answer] → [Belief Q2] → [Answer]
```

## Saving Results

In [11]:
def save_records(game_name, records, model=MODEL):
    """Save records including beliefs to JSON."""
    timestamp = datetime.now().strftime("%Y_%m_%d-%I_%M_%S_%p")
    file_name = f'{game_name}_{model}_{timestamp}.json'
    os.makedirs('records', exist_ok=True)
    with open(os.path.join('records', file_name), 'w') as f:
        json.dump(records, f, indent=2, default=str)
    print(f"Saved to records/{file_name}")

# Example:
# save_records('pd', records_pd)
# save_records('stag_hunt', records_sh)
# save_records('beauty_contest', records_bc)
# save_records('auction', records_auc)
# save_records('ultimatum', records_ult)